In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import plotly.graph_objects as go
from tqdm import tqdm
tqdm.pandas()
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#define the data folder location
project_path = Path(os.getcwd())
data_path = project_path / 'data'

# Introduction

This notebook guides you through the analysis of the ISG screen performed using the Incucyte live-cell imaging system. The fluorescence signal (Green Integrated Intensity / Red Integrated Intensity, GII/RII) is used as a proxy for viral replication across 7 viruses:

- SARS-CoV-2
- HSV-1
- MeV
- RVFV
- SFV
- VACV
- VSV

The 300 ISGs tested were divided into 4 sets (plates), reflected in the filenames as `S<set number>`. Each set contains 5 NTCs used to normalise within-plate values.

Filenames follow the convention:

*V<span style="color:blue">\<virus number\></span>\_S<span style="color:blue">\<set number\></span>\_R<span style="color:blue">\<biological replicate number\></span>.txt*

> **Note:** For SARS-CoV-2, biological replicates are labelled 3, 4, 6, and 7 instead of 1–4.


## Definitions 

Below are the definitions and mappings used throughout this analysis.


In [ ]:
# Non-Targeting Controls (NTCs): wells expressing a non-targeting guide RNA,
# used as the within-plate baseline for normalisation.
NTCs = ['NTC1', 'NTC2', 'NTC3', 'NTC4', 'NTC5']


and mapping the virus code from the names of the incucyte files to the actual virus name:

In [ ]:
id2virus={
"0"	: "mock",
"1"	: "HSV-1",	
"2"	: "MeV",
"3"	: "RVFV",
"4"	: "VACV",
"5"	: "YFV",
"6"	: "VSV",	
"7"	: "SFV",
'8'	: 'SARS-CoV-2',
}

# **Viability**

In [ ]:
from src.read_data import read_viability_data


## Read the viability data

`read_viability_data` reads the Incucyte viability measurements (Green Integrated Intensity, GII) from mock-infected control plates and structures them as a multi-index DataFrame.


In [ ]:
# define where the viability data is
viability_path = data_path / 'viability'

In [ ]:
#read the viability data
viab_df=read_viability_data(viability_path)
viab_df.head()

## Fit the viability data

In [ ]:
from src.fit_data import exponential_fit

`fit_exp_viability` fits the viability GII signal acording to the following equation

$$
  viability(t) = a\cdot e^{b \cdot t}
$$

and extract the parameters ${a}$ & ${b}$ as the *initial_GII* and *growth_rate* respectively

In [ ]:
#apply the curve fitting to the viability data using an exponential model
viab_fit_full_df=viab_df.apply(exponential_fit).T
viab_fit_full_df.head()

The fit produces a DataFrame with parameters `a` (`starting_GII`) and `b` (`growth_rate`), together with goodness-of-fit metrics:

* `r_squared` — coefficient of determination (R²)
* `r_squared_adj` — adjusted R²


## Define Toxicity:

A run is considered bad (Toxic) if it fulfills the following criteria:
1. $${starting\_GII < 0.7 \cdot 10^{6}}$$

<p style="text-align: center;">which controls for bad initial cell count due to cell toxicity</p>


2. $${r^2\_adj < 0.9}$$
<p style="text-align: center;">which controls for bad fitting due to decreasing (rather than the desired increasing) GII because of cell death</p>



In [ ]:
# A run is flagged as toxic when the fit quality is poor or the initial cell count is too low.
# NaN starting_GII corresponds to a completely failed fit (no signal or extreme noise).
viab_fit_full_df['is_toxic'] = (
    (viab_fit_full_df['r_squared_adj'] < 0.9)
    | (viab_fit_full_df['starting_GII'] < 0.7e6)
    | viab_fit_full_df['starting_GII'].isna()
)


If a KO is toxic in 3 or more replicates then the KO is generally toxic


In [ ]:
toxic_kos = viab_fit_full_df.query('is_toxic').groupby('knockout')[['is_toxic']].count().query('is_toxic>=3').index 

In [ ]:
# the following are the KO that are toxic in at least 3 replicates
for ko in toxic_kos:
    print(ko)

In [ ]:
#filter out the toxic KOs and also the bad viability replicates to avoid biassing the mean (1 replicate from POLI is toxic/bad initial signal)
viab_fit_df=viab_fit_full_df.query('~knockout.isin(@toxic_kos)').query('~is_toxic')[['starting_GII','growth_rate']]
viab_fit_df.head()

Here is an example of how the exponential fit looks like:

In [ ]:
from src.fit_data import exp_func

col = ('V0_S1_R3', 'EPHB2')

exponential_fit_fig = go.Figure()
exponential_fit_fig.add_trace(go.Scatter(
    x=viab_df.loc[:, col].dropna().index,
    y=viab_df.loc[:, col].dropna(),
    line={"width": 4},
    name='original signal',
))

a, b = viab_fit_df.loc[col, ['starting_GII', 'growth_rate']].values
t = np.linspace(0, 70, 100)
y = exp_func(t, a, b)
exponential_fit_fig.add_trace(go.Scatter(
    x=t,
    y=y,
    line={'dash': 'dashdot', 'width': 4, 'color': 'black'},
    name='fitted signal',
))
exponential_fit_fig.update_layout(
    title='Exponential fit example',
    yaxis_title='GII (a.u.)',
    xaxis_title='time (h)',
)


## Normalization

#### Average the NTCs in each plate

This average will be used to normalize each plate

In [ ]:
avg_NTC_viab_per_plate_df = viab_fit_df.query('knockout.isin(@NTCs)').groupby('mock_plate_id').mean()
avg_NTC_viab_per_plate_df


#### Normalize the KOs to the mean of NTCs in the same plate

In [ ]:
normalized_viab_fit_df = viab_fit_df / avg_NTC_viab_per_plate_df
normalized_viab_fit_df.head()


#### Average the normalized starting GII and growth rate for each KO across the different biological replicates (`mock_plate_id`)

In [ ]:
avg_normalized_viab_fit_df=normalized_viab_fit_df.groupby('knockout').mean()
avg_normalized_viab_fit_df.head()

-----------------------------

# **Fluorescence**


In [ ]:
flrsc_path = data_path / 'fluorescence'

## Read the fluorescence data

In [ ]:
from src.read_data import read_flrsc_data

`read_flrsc_data` reads the Incucyte fluorescence reporter measurements (GII/RII) and structures them as a multi-index DataFrame keyed by virus, set, biological replicate, matched mock plate, and knockout identity.


In [ ]:
flrsc_raw_df=read_flrsc_data(flrsc_path,id2virus)
flrsc_raw_df.head()

## Data cleaning

In [ ]:
from src.process_signal import smooth, remove_initial_noise


#### 1. Drop the toxic KOs that were determined from the viability screen

In [ ]:
flrsc_df=flrsc_raw_df.drop(toxic_kos,axis=1,level='knockout')
flrsc_df.head()

#### 2. Remove initial noise

In some cases there is an intial spike at very early time points that we remove. This spike is due to background fluorescence that gets normalized after the main fluorescence signal dominates. 

The following code removes this initial spike from the fluorescence signal assuming that this spike exist at t<=9h.

The graph below demonstrates the signal before and after the initial noise removal.

In [ ]:
no_init_noise_flrsc_data_df=flrsc_df.apply(remove_initial_noise)

In [ ]:
from plotly.subplots import make_subplots

# Create subplot figure
spike_fig = make_subplots(rows=1, cols=2, subplot_titles=("raw signal", "initial spike removed"))

# Add traces to the subplots
col=( 'SARS-CoV-2', '1', '3', np.nan, 'CTAGE1')

spike_fig.add_trace(go.Scatter(x=flrsc_df.loc[:6,col].dropna().index,y=flrsc_df.loc[:6,col].dropna(),marker=dict(color='red'),line={"dash":'dot'},name='initial spike'),row=1, col=1)
spike_fig.add_trace(go.Scatter(x=flrsc_df.loc[6:,col].dropna().index,y=flrsc_df.loc[6:,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=1)
spike_fig.add_trace(go.Scatter(x=no_init_noise_flrsc_data_df.loc[:6,col].dropna().index,y=no_init_noise_flrsc_data_df.loc[:6,col].dropna(),showlegend=False,marker=dict(color='red'),name='initial spike'),row=1, col=2)
spike_fig.add_trace(go.Scatter(x=no_init_noise_flrsc_data_df.loc[6:,col].dropna().index,y=no_init_noise_flrsc_data_df.loc[6:,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=2)

spike_fig.update_xaxes(title_text="time (h)", row=1, col=1)
spike_fig.update_yaxes(title_text="fluorescence (GII/RII)", row=1, col=1)

spike_fig.update_xaxes(title_text="time (h)", row=1, col=2)
spike_fig.update_yaxes(title_text="fluorescence (GII/RII)", row=1, col=2)
# Show figure
spike_fig.show()

#### 3. Remove transient noise

In some signals a sharp transient noise exists due to misfocosed measurement. This is corrected using a smoothing function consisting of a median filter (kernel=3) followed by a savgol filter

In [ ]:
smoothed_flrsc_data_df = no_init_noise_flrsc_data_df.apply(lambda col: smooth(col, median_kernel=3))


The following graph demonstrates the effect of the smoothing function

In [ ]:

smoothed_fig = make_subplots(rows=1, cols=2, subplot_titles=("raw signal", "noise smoothed"))

# Add traces to the subplots

col=( 'SFV', '2', '3', 'V0_S2_R5', 'IFI44')
smoothed_fig.add_trace(go.Scatter(x=no_init_noise_flrsc_data_df.loc[:51,col].dropna().index,y=no_init_noise_flrsc_data_df.loc[:51,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=1)
smoothed_fig.add_trace(go.Scatter(x=no_init_noise_flrsc_data_df.loc[51:58,col].dropna().index,y=no_init_noise_flrsc_data_df.loc[51:58,col].dropna(),marker=dict(color='red'),line={"dash":'dot'},name='noise'),row=1, col=1)
smoothed_fig.add_trace(go.Scatter(x=no_init_noise_flrsc_data_df.loc[57:,col].dropna().index,y=no_init_noise_flrsc_data_df.loc[57:,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=1)
smoothed_fig.add_trace(go.Scatter(x=smoothed_flrsc_data_df.loc[:51,col].dropna().index,y=smoothed_flrsc_data_df.loc[:51,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=2)
smoothed_fig.add_trace(go.Scatter(x=smoothed_flrsc_data_df.loc[51:58,col].dropna().index,y=smoothed_flrsc_data_df.loc[51:58,col].dropna(),showlegend=False,marker=dict(color='red'),name='noise'),row=1, col=2)
smoothed_fig.add_trace(go.Scatter(x=smoothed_flrsc_data_df.loc[57:,col].dropna().index,y=smoothed_flrsc_data_df.loc[57:,col].dropna(),marker=dict(color='blue'),showlegend=False),row=1, col=2)

smoothed_fig.update_xaxes(title_text="time (h)", row=1, col=1)
smoothed_fig.update_yaxes(title_text="fluorescence (GII/RII)", row=1, col=1)

smoothed_fig.update_xaxes(title_text="time (h)", row=1, col=2)
smoothed_fig.update_yaxes(title_text="fluorescence (GII/RII)", row=1, col=2)
# Show figure
smoothed_fig.show()

## Fit the fluorescence data

In [ ]:
from src.fit_data import sigmoid_fit

`sigmoid_fit` fits the fluorescence signal ${\frac{\int GII}{\int RII}(t)}$ acording to the following equation

$$
  \frac{\int GII}{\int RII}(t) \approx sigmoid(t) = c + \frac{K}{1 + e^{-\beta(t-\tau)}}
$$

and extract the parameters ${c}$, ${K}$, ${\beta}$ & ${\tau}$ 

In [ ]:
flrsc_fit_raw_df = smoothed_flrsc_data_df.progress_apply(sigmoid_fit).T
flrsc_fit_raw_df.head()

filter out runs that couldn't be fit

In [ ]:
failed_fit_df=flrsc_fit_raw_df.loc[flrsc_fit_raw_df.isna().all(axis=1)]
flrsc_fit_df=flrsc_fit_raw_df.drop(failed_fit_df.index)
failed_fit_df

Here is an example of how a fit looks like:

In [ ]:
from src.fit_data import sigmoid

col=('YFV', '2', '3', 'V0_S2_R4', 'FGL2')

sigmoid_fit_fig=go.Figure()
sigmoid_fit_fig.add_trace(go.Scatter(x=smoothed_flrsc_data_df.loc[:,col].dropna().index,
                                     y=smoothed_flrsc_data_df.loc[:,col].dropna(),
                                     line={"width":4},
                                     name='original (cleaned) signal'))

c,K,beta,tau=flrsc_fit_df.loc[col,['c','K','beta','tau']].values
t=np.linspace(0,38,100)
y=sigmoid(t,K,tau,beta,c)
sigmoid_fit_fig.add_trace(go.Scatter(x=t,
                                     y=y,
                                     line={'dash':'dashdot','width':4,'color':'black'},
                                     name='fitted signal'))
sigmoid_fit_fig.update_layout(title='sigmoid fit example',yaxis_title='fluorescence (GII/RII)',xaxis_title='time (h)')


## Normalization

#### average the NTCs in each plate

In [ ]:
avg_NTC_flrsc_per_plate_df = flrsc_fit_df[['K', 'tau', 'beta', 'c']].query('knockout.isin(@NTCs)').groupby(['virus', 'set', 'bio_rep']).mean()
avg_NTC_flrsc_per_plate_df.head()


#### normalize the KOs to the mean of NTCs in the same plate

In [ ]:
normalized_flrsc_fit_df = flrsc_fit_df[['K', 'tau', 'beta', 'c']] / avg_NTC_flrsc_per_plate_df
normalized_flrsc_fit_df.head()


#### Average the normalized sigmoid function parameters for each KO across the 4 biological replicates (bio_rep)

In [ ]:
avg_normalized_flrsc_fit_df=normalized_flrsc_fit_df.groupby(['virus','set','knockout']).mean()
avg_normalized_flrsc_fit_df.head()


# Merge the viability and fluorescence data

## unnormalized

In [ ]:
flrsc_fit_df[['K','tau','beta','c']].reset_index().merge(viab_fit_df.reset_index(),how='left').set_index(flrsc_fit_df.index.names)

## normalized

In [ ]:
normalized_params_full_df=avg_normalized_flrsc_fit_df.merge(avg_normalized_viab_fit_df,left_index=True,right_index=True,how='left')
normalized_params_full_df